#1. Carga de datos

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

esqueleto = pd.read_parquet('/content/drive/MyDrive/TFM/favorita_panel_completo.parquet')
esqueleto.shape

Mounted at /content/drive


(6879586, 19)

#2. Análisis descriptivo complementario
###2.1 Sobredispersión

In [2]:
# Se calcula la media y la varianza de unit_sales para cada una de las 4078 series
# Si la varianza es mayor que la media en la mayoría de las series, hay sobredispersión y justifica la binomial negativa
stats_por_serie = esqueleto.groupby(['store_nbr', 'item_nbr'])['unit_sales'].agg(['mean', 'var']).reset_index()
stats_por_serie['sobredispersa'] = stats_por_serie['var'] > stats_por_serie['mean']

print('Series sobredispersas (varianza > media):', stats_por_serie['sobredispersa'].sum(), 'de', len(stats_por_serie))
print('Porcentaje:', round(stats_por_serie['sobredispersa'].mean() * 100, 1), '%')
stats_por_serie[['mean', 'var']].describe()

Series sobredispersas (varianza > media): 4057 de 4078
Porcentaje: 99.5 %


,mean,var
count,4078.000000,4078.000000
mean,5.382753,49.429646
std,7.180786,247.852707
min,0.000593,0.000593
25%,1.299348,4.597326
50%,3.215768,13.231350
75%,6.729105,39.074954
max,108.055717,7425.765625


La media de las medias es de 5.3 mientras que la media de las varianzas es de 49.4, con lo cual se descarta una distribución normal con verosimilitud y se confirma que la binomial negativa es la distribución correcta para aplicar en la capa de salida del modelo.

###2.2 Asimetría de volumen entre series

Volumen promedio de venta por serie, ordenado de mayor a menor.

In [3]:
# Se Compara el promedio del 10% de series con más venta contra el 10% con menos,
# para ver cuán desigual es el volumen entre combinaciones tienda-producto.
volumen_por_serie = esqueleto.groupby(['store_nbr', 'item_nbr'])['unit_sales'].mean().sort_values(ascending=False)

n_decil = int(len(volumen_por_serie) * 0.1)
print('Volumen promedio, 10% de series con MÁS venta:', volumen_por_serie.head(n_decil).mean())
print('Volumen promedio, 10% de series con MENOS venta:', volumen_por_serie.tail(n_decil).mean())
print('Razón entre ambos extremos:', volumen_por_serie.head(n_decil).mean() / volumen_por_serie.tail(n_decil).mean())
volumen_por_serie.describe()

Volumen promedio, 10% de series con MÁS venta: 21.25691
Volumen promedio, 10% de series con MENOS venta: 0.2503623
Razón entre ambos extremos: 84.904594


,unit_sales
count,4078.000000
mean,5.382753
std,7.180781
min,0.000593
25%,1.299348
50%,3.215768
75%,6.729105
max,108.055717


10% de las series con mñas venta promedian 21.3 mientras que el 10$ de las series con menos ventas promedian 0.25. Se debe aplicar una corrección para que el modelo no se ajuste a las series de alto volumen e ignore a las de bajo volumen. Se respalda el muestreo ponderado.

###2.3 Cold-start
Búsqueda de series que entraron después del inicio del dataset.

In [4]:
#Carga del archivo con transacciones reales para revisar cuándo ocurrió la primera venta real de cada
#combinación tienda-producto
df_transacciones = pd.read_parquet('/content/drive/MyDrive/TFM/favorita_filtrado.parquet')

primera_venta = df_transacciones.groupby(['store_nbr', 'item_nbr'])['date'].min().reset_index()
primera_venta.columns = ['store_nbr', 'item_nbr', 'primera_fecha_venta']

fecha_inicio_dataset = df_transacciones['date'].min()
primera_venta['dias_de_retraso'] = (primera_venta['primera_fecha_venta'] - fecha_inicio_dataset).dt.days

print(primera_venta['dias_de_retraso'].describe())
print('Series que iniciaron más de 90 días después del inicio del dataset:', (primera_venta['dias_de_retraso'] > 90).sum())
print('Series que iniciaron más de 365 días después:', (primera_venta['dias_de_retraso'] > 365).sum())

count    4078.000000
mean      506.832271
std       510.509264
min         0.000000
25%         0.000000
50%       307.000000
75%       909.000000
max      1680.000000
Name: dias_de_retraso, dtype: float64
Series que iniciaron más de 90 días después del inicio del dataset: 2686
Series que iniciaron más de 365 días después: 1641


- 2686 productos (66%) tienen su primera venta después de 90 días del inicio del dataset y 1641 (40%) un año después.

Búsqueda de causa del retraso (producto nuevo, surtido, apertura de tienda)

In [5]:
# Apertura aparente de cada tienda: su primera transacción registrada, sin importar el producto
primera_venta_tienda = df_transacciones.groupby('store_nbr')['date'].min().reset_index()
primera_venta_tienda.columns = ['store_nbr', 'apertura_aparente_tienda']
primera_venta_tienda['dias_apertura_tienda'] = (primera_venta_tienda['apertura_aparente_tienda'] - fecha_inicio_dataset).dt.days

print('Tiendas cuya primera transacción registrada ocurre >90 días después del inicio del dataset:')
print(primera_venta_tienda[primera_venta_tienda['dias_apertura_tienda'] > 90])

# Primera venta de cada producto en cualquier tienda de la cadena (lanzamiento global del producto)
primera_venta_producto_global = df_transacciones.groupby('item_nbr')['date'].min().reset_index()
primera_venta_producto_global.columns = ['item_nbr', 'primera_venta_global']

# Cruzar con el diagnóstico por serie (store, item) para clasificar cada retraso
primera_venta = primera_venta.merge(primera_venta_tienda, on='store_nbr', how='left')
primera_venta = primera_venta.merge(primera_venta_producto_global, on='item_nbr', how='left')

# Clasificación:
# - tienda_recien_abierta: la tienda misma abrió tarde, el retraso no es del producto
# - producto_nuevo_global: el producto es nuevo en toda la cadena (lanzamiento genuino)
# - decision_surtido: el producto ya existía en otras tiendas mucho antes; esta tienda solo tardó en tenerlo
primera_venta['tienda_recien_abierta'] = primera_venta['dias_apertura_tienda'] > 30
primera_venta['producto_nuevo_global'] = (primera_venta['primera_fecha_venta'] - primera_venta['primera_venta_global']).dt.days <= 7
primera_venta['decision_surtido'] = (~primera_venta['tienda_recien_abierta']) & (~primera_venta['producto_nuevo_global'])

print('\nDistribución de causas entre las series con retraso > 90 días:')
retrasadas = primera_venta[primera_venta['dias_de_retraso'] > 90]
print('Tienda recién abierta:', retrasadas['tienda_recien_abierta'].sum())
print('Producto nuevo global:', retrasadas['producto_nuevo_global'].sum())
print('Decisión de surtido (producto ya existía en otras tiendas):', retrasadas['decision_surtido'].sum())

Tiendas cuya primera transacción registrada ocurre >90 días después del inicio del dataset:
    store_nbr apertura_aparente_tienda  dias_apertura_tienda
12         20               2015-02-13                   772

Distribución de causas entre las series con retraso > 90 días:
Tienda recién abierta: 199
Producto nuevo global: 1859
Decisión de surtido (producto ya existía en otras tiendas): 667


- La tienda 20 tiene apertura tardía y explica 199 de las 2686 series retrasadas
- 1859 series (69%) corresponden a lanzamientos de productos en toda la cadena
- 667 series (25%) corresponden a decisiones de surtido entre tiendas

Historial útil después de truncar las series

In [6]:
fecha_fin_dataset = df_transacciones['date'].max()

primera_venta['historia_restante_dias'] = (fecha_fin_dataset - primera_venta['primera_fecha_venta']).dt.days

print(primera_venta['historia_restante_dias'].describe())
print('\nSeries con menos de 30 días de historia útil:', (primera_venta['historia_restante_dias'] < 30).sum())
print('Series con menos de 60 días de historia útil:', (primera_venta['historia_restante_dias'] < 60).sum())
print('Series con menos de 90 días de historia útil:', (primera_venta['historia_restante_dias'] < 90).sum())
print('Series con menos de 180 días de historia útil:', (primera_venta['historia_restante_dias'] < 180).sum())

count    4078.000000
mean     1179.167729
std       510.509264
min         6.000000
25%       777.000000
50%      1379.000000
75%      1686.000000
max      1686.000000
Name: historia_restante_dias, dtype: float64

Series con menos de 30 días de historia útil: 3
Series con menos de 60 días de historia útil: 8
Series con menos de 90 días de historia útil: 36
Series con menos de 180 días de historia útil: 168


- La mediana del historial restante es de 1379 días
- el 75% de los datos conservan al menos 777 días útiles de historial
- 36 series tienen menos de 90 días de historial y 168 menos de 180 días, lo cual es manejable dejándolas fuera o excluyéndolas del entrenamiento.

#3. Correcciones
###3.1 Eliminar el relleno de ceros en pre-lanzamiento

In [7]:
# contexto de Lim et al. (90) + horizonte real de la competencia Favorita (16)
UMBRAL_MINIMO = 106

#Series a excluir por historia insuficiente
series_excluidas = primera_venta[primera_venta['historia_restante_dias'] < UMBRAL_MINIMO][['store_nbr', 'item_nbr']]
print('Series excluidas por historia insuficiente (<', UMBRAL_MINIMO, 'días):', len(series_excluidas))
print('Series retenidas:', len(primera_venta) - len(series_excluidas))

#Panel corregido: se cruza el panel completo con la primera venta real de cada serie
panel_corregido = esqueleto.merge(
    primera_venta[['store_nbr', 'item_nbr', 'primera_fecha_venta', 'historia_restante_dias']],
    on=['store_nbr', 'item_nbr'],
    how='left'
)

# Se descartan las series con historia insuficiente
panel_corregido = panel_corregido[panel_corregido['historia_restante_dias'] >= UMBRAL_MINIMO]

# Se trunca cada serie: fuera las filas anteriores a su primera venta real (el relleno fantasma)
filas_antes = len(panel_corregido)
panel_corregido = panel_corregido[panel_corregido['date'] >= panel_corregido['primera_fecha_venta']]
filas_despues = len(panel_corregido)

print('Filas antes del truncado:', filas_antes)
print('Filas después del truncado:', filas_despues)
print('Filas eliminadas por relleno fantasma pre-lanzamiento:', filas_antes - filas_despues)

# 3. Covariable de edad (DeepAR, Salinas et al. 2020): días desde la primera venta real de cada serie
panel_corregido['edad'] = (panel_corregido['date'] - panel_corregido['primera_fecha_venta']).dt.days

panel_corregido[['store_nbr', 'item_nbr', 'date', 'unit_sales', 'edad']].head()

Series excluidas por historia insuficiente (< 106 días): 97
Series retenidas: 3981
Filas antes del truncado: 6715947
Filas después del truncado: 4804366
Filas eliminadas por relleno fantasma pre-lanzamiento: 1911581


,store_nbr,item_nbr,date,unit_sales,edad
0,1,122095,2013-01-02,1.0,0
1,1,122095,2013-01-03,1.0,1
2,1,122095,2013-01-04,2.0,2
3,1,122095,2013-01-05,1.0,3
4,1,122095,2013-01-06,1.0,4


In [8]:
# Verificación
print('Series únicas en el panel corregido:', panel_corregido.groupby(['store_nbr', 'item_nbr']).ngroups)
print('Edad mínima:', panel_corregido['edad'].min(), '| Edad máxima:', panel_corregido['edad'].max())

# Se confirma sobre la serie con mayor retraso original que ya no queden ceros fantasma
ejemplo = primera_venta.sort_values('dias_de_retraso', ascending=False).iloc[0]
serie_ejemplo = panel_corregido[
    (panel_corregido['store_nbr'] == ejemplo['store_nbr']) &
    (panel_corregido['item_nbr'] == ejemplo['item_nbr'])
]
print('\nStore', ejemplo['store_nbr'], '/ item', ejemplo['item_nbr'])
print('Primera venta real registrada:', ejemplo['primera_fecha_venta'])
print('Primera fecha presente en el panel corregido:', serie_ejemplo['date'].min() if len(serie_ejemplo) else 'excluida por historia insuficiente')

Series únicas en el panel corregido: 3981
Edad mínima: 0 | Edad máxima: 1686

Store 3 / item 2028307
Primera venta real registrada: 2017-08-09 00:00:00
Primera fecha presente en el panel corregido: excluida por historia insuficiente


In [9]:
# Guardar
panel_corregido.to_parquet('/content/drive/MyDrive/TFM/favorita_panel_corregido.parquet', index=False)
print('Guardado:', panel_corregido.shape)

Guardado: (4804366, 22)


In [10]:
# Verificación alternativa: serie con mayor retraso entre las que sí se conservaron
retenidas = primera_venta[primera_venta['historia_restante_dias'] >= UMBRAL_MINIMO]
ejemplo2 = retenidas.sort_values('dias_de_retraso', ascending=False).iloc[0]

serie_ejemplo2 = panel_corregido[
    (panel_corregido['store_nbr'] == ejemplo2['store_nbr']) &
    (panel_corregido['item_nbr'] == ejemplo2['item_nbr'])
]
print('Store', ejemplo2['store_nbr'], '/ item', ejemplo2['item_nbr'])
print('Retraso original:', ejemplo2['dias_de_retraso'], 'días')
print('Primera venta real registrada:', ejemplo2['primera_fecha_venta'])
print('Primera fecha presente en el panel corregido:', serie_ejemplo2['date'].min())
print('¿Coinciden exactamente?:', serie_ejemplo2['date'].min() == ejemplo2['primera_fecha_venta'])

Store 2 / item 2053590
Retraso original: 1561 días
Primera venta real registrada: 2017-04-12 00:00:00
Primera fecha presente en el panel corregido: 2017-04-12 00:00:00
¿Coinciden exactamente?: True


- se eliminó el 28,5% de las filas del panel por ser rellenos fantasmas de ceros (los que se rellenaron antes de que exista la primera venta real)